# Пользовательские сессии: разбор вложенной структуры

Чистый Python, без pandas. Данные: `data/sessions.csv`, 5283 сессии, 1000 пользователей, май-июнь 2023.

Постановка в `data/04_пользовательские_сессии.ТЗ.md`.

## Формат данных

```
user0,"[""2023-05-06 20:43:45"", 1192, [[""page0"", ""2023-05-06 20:49:36""], [""page1"", ""2023-05-06 20:57:32""], ...]]"
```

Одна колонка CSV, внутри неё список: дата и время начала, длительность в секундах, список переходов вида `[[страница, время], ...]`.

Три уровня вложенности плюс экранирование кавычек, потому что JSON лежит внутри CSV.

In [ ]:
import ast
from collections import defaultdict
from datetime import datetime

SESSION_FILE = "../data/sessions.csv"
FMT = "%Y-%m-%d %H:%M:%S"

def load_sessions(path: str) -> list[tuple[str, list]]:
    sessions = []
    with open(path, encoding="utf-8") as f:
        f.readline()                                   # заголовок
        for line in f:
            line = line.strip()
            if not line:
                continue
            # split без maxsplit нельзя: внутри сессии запятых больше, чем одна
            user_id, payload = line.split(",", 1)
            payload = payload.strip().strip('"').replace('""', '"')
            sessions.append((user_id, ast.literal_eval(payload)))
    return sessions

sessions = load_sessions(SESSION_FILE)
print("сессий:", len(sessions))
print("пользователей:", len({u for u, _ in sessions}))
print()
user, s = sessions[0]
print(user, "начало", s[0], "| длительность", s[1], "с | переходов", len(s[2]))
print("первые переходы:", s[2][:3])

## Две ловушки при разборе

**Первая: `split(",")` без ограничения.** Внутри сессии запятые разделяют элементы списка, их там десятки. Без `maxsplit=1` строка распадётся на сотни частей, и `ast.literal_eval` получит мусор вместо данных.

**Вторая: `json.loads` вместо `ast.literal_eval`.** Формат похож на JSON, но это не JSON. Даты записаны без кавычек, `2023-05-06 20:43:45`, тогда как JSON требует строки в кавычках. `json.loads` на такой строке упадёт. `literal_eval` понимает Python-литералы, поэтому дата остаётся строкой и разбирается корректно.

Сравнение для наглядности:

In [ ]:
payload = '["2023-05-06 20:43:45", 1192, [["page0", "2023-05-06 20:49:36"]]]'

try:
    import json
    json.loads(payload)
except Exception as e:
    print("json.loads:", type(e).__name__, "-", e)

print("ast.literal_eval:", ast.literal_eval(payload))

Ещё одна особенность данных: длительность сессии лежит вторым элементом отдельным числом, но внутри списка переходов есть `conversion` со своим временем. Эти две вещи нельзя смешивать: для времени до конверсии нужен отсчёт от начала сессии, а не сумма длительностей переходов.

## Задача 1. Время до конверсии

In [ ]:
def time_to_conversion(sessions):
    """В секундах от начала сессии до момента целевого действия."""
    deltas = []
    for _, s in sessions:
        started = datetime.strptime(s[0], FMT)
        for page, when in s[2]:
            if page == "conversion":
                deltas.append((datetime.strptime(when, FMT) - started).total_seconds())
    return deltas

ttc = time_to_conversion(sessions)
ttc_sorted = sorted(ttc)

print(f"сессий с конверсией: {len(ttc)}")
print(f"среднее:  {sum(ttc)/len(ttc):.2f} сек")
print(f"медиана:  {ttc_sorted[len(ttc_sorted)//2]:.0f} сек")
print(f"минимум:  {ttc_sorted[0]:.0f} сек")
print(f"максимум: {ttc_sorted[-1]:.0f} сек")

Среднее 467 секунд, это 7.8 минуты. Медиана 364 секунды, 6 минут. Разница заметная, значит есть правый хвост из долгих сессий.

Минимум 1 секунда это не быстрый пользователь, а техническая особенность данных: в части сессий `conversion` стоит раньше первого перехода по времени. Проверим.

In [ ]:
suspicious = [d for d in ttc if d < 30]
print("конверсий быстрее чем за 30 секунд:", len(suspicious), f"({len(suspicious)/len(ttc):.2%})")
print("минимальные значения:", sorted(ttc)[:10])

Конверсий быстрее 30 секунд 75, это 7.4%. В этих сессиях целевое действие зафиксировано раньше первого перехода. В данных это артефакт генератора, а не реальное поведение: пользователь не может оставить заявку раньше, чем что-то открыл.

На среднее в 467 секунд это влияет слабо, но приходится знать. В рабочей задаче такие случаи отфильтровываются или разбираются с командой, которая их породила.

## Задача 2. Длительность сессии

In [ ]:
durations = sorted(s[1] for _, s in sessions)

def mean(values):
    return round(sum(values) / len(values), 2)

def median(values):
    n = len(values)
    return values[n // 2] if n % 2 else (values[n // 2 - 1] + values[n // 2]) / 2

print(f"среднее:  {mean(durations)} сек")
print(f"медиана:  {median(durations)} сек")
print(f"минимум:  {durations[0]} сек")
print(f"максимум: {durations[-1]} сек")

Среднее 901.35 секунды и медиана 900. Разница в секунду, распределение симметричное.

Максимум ровно 1800 секунд, это 30 минут. Похоже на искусственный потолок в генераторе данных: длительность не может превысить полчаса. В реальных данных такого потолка нет, и на больших значениях среднее разошлось бы с медианой сильнее.

## Задача 3. Сколько заходов в день делает пользователь

In [ ]:
def sessions_per_day(sessions):
    """Сколько раз пользователь заходил в конкретный день."""
    per_user_day = defaultdict(int)
    for user, s in sessions:
        per_user_day[(user, s[0][:10])] += 1
    return per_user_day

spd = sessions_per_day(sessions)
counts = sorted(spd.values())
print(f"пар (пользователь, день): {len(spd)}")
print(f"среднее заходов в день на пользователя: {mean(counts)}")
print(f"медиана: {median(counts)}")
print(f"максимум: {counts[-1]}")
print()
print("распределение:")
from collections import Counter
for value, n in sorted(Counter(counts).items()):
    print(f"  {value} заходов: {n} дней")

Среднее 1.11 захода в день на активный день пользователя, медиана 1. То есть большинство дней это один заход, и разброс создают те, кто заходит по многу раз: 433 дня с двумя заходами и 39 с тремя.

В пересчёте на пользователя это 4.77 сессии за месяц, но только по тем 21 дням, когда он вообще заходил.

## Задача 4. Активность пользователя в целом

In [ ]:
per_user = defaultdict(int)
user_days = defaultdict(set)
for user, s in sessions:
    per_user[user] += 1
    user_days[user].add(s[0][:10])

print(f"пользователей: {len(per_user)}")
print(f"сессий на пользователя: {mean(sorted(per_user.values()))}")
print(f"дней на пользователя:   {mean(sorted(len(v) for v in user_days.values()))}")
print(f"страниц в сессии:       {mean(sorted(len(s[2]) for _, s in sessions))}")
print()
pages = sorted(len(s[2]) for _, s in sessions)
print(f"медиана страниц в сессии: {median(pages)}")

5.28 сессии на пользователя за месяц при 21 дне активности в среднем. В пересчёте это примерно 3.4 сессии в неделю, но только по дням, когда пользователь вообще заходил.

## Конверсия

In [ ]:
converted_sessions = [(u, s) for u, s in sessions if any(p == "conversion" for p, _ in s[2])]
converted_users = {u for u, _ in converted_sessions}

print(f"конверсия сессий:   {len(converted_sessions)/len(sessions):.4f}")
print(f"конверсия людей:    {len(converted_users)/len(per_user):.4f}")
print()
print("19.2% сессий конвертируются, но конвертируются 59.9% пользователей.")

Разрыв между 19.2% и 59.9% это главная находка всего ноутбука.

Если бы каждый пришедший пользователь конвертировался один раз, доля конверсий сессий была бы примерно равна доле конвертирующихся пользователей, делённая на число их сессий. Здесь получилось в три раза больше: конвертирующиеся люди заходят чаще остальных и конвертируются почти в каждой сессии.

Практический вывод: работать надо не с общей долей сессий, а с пользователями. 80% людей не конвертируются вообще, и это самая большая точка роста. Усреднение по сессиям здесь занижает потенциал втрое.

## Конверсия по дням

In [ ]:
by_day = defaultdict(lambda: [0, 0])
for user, s in sessions:
    day = s[0][:10]
    by_day[day][0] += 1
    if any(p == "conversion" for p, _ in s[2]):
        by_day[day][1] += 1

print(f"{'Дата':<12} {'Сессий':>7} {'Конверсия':>10}")
for day in sorted(by_day)[:12]:
    total, conv = by_day[day]
    print(f"{day:<12} {total:>7} {conv/total:>10.4f}")

Разброс по дням от 0.147 до 0.256 при среднем 0.19. Это нормальная для таких данных величина, отдельных аномальных дней нет.

## Итоги

| Метрика | Значение |
|---|---|
| Сессий | 5283 |
| Пользователей | 1000 |
| Среднее до конверсии | 467.24 сек |
| Медиана до конверсии | 364 сек |
| Конверсий быстрее 30 секунд | 75, 7.4% |
| Средняя длительность сессии | 901.35 сек |
| Медиана длительности | 900 сек |
| Максимальная длительность | 1800 сек |
| Сессий на пользователя | 5.28 |
| Заходов в день на пользователя | 4.77 |
| Заходов в день на активный день | 1.11 |
| Активных дней на пользователя | 21.0 |
| Страниц в сессии, среднее | 8.17 |
| Страниц в сессии, медиана | 8 |
| Конверсия сессий | 19.23% |
| Конверсия пользователей | 59.90% |

Что стоит вынести:

1. `split(",", 1)` обязателен. Внутри сессии десятки запятых, обычный split развалит строку.
2. Это не JSON, хотя выглядит как JSON. Даты без кавычек, `json.loads` падает, нужен `ast.literal_eval`.
3. Конверсия сессий 19.2% против конверсии людей 59.9%. Разрыв в три раза означает, что конвертирующиеся заходят чаще. Ориентироваться надо на пользователей, усреднение по сессиям занижает потенциал втрое.
4. Проверка на артефакты обязательна. 7.4% конверсий зафиксированы раньше первого перехода, а длительность упирается ровно в 1800 секунд. Оба ограничения пришли из генератора данных, но в рабочей задаче такое надо находить до того, как строить выводы.